# 10 — MDM_InsertDateTime evidence audit

Maksim Pikalov, 2026-09-30. Read-only inspection of both supplied exports and the
local integrated-review-v1 partitions. No timestamp conversion rule is applied.

**Scope:** all records, plus a separately reported subset without a current
quarantine decision. Missing means blank/whitespace or case-insensitive `null`.
One CSV record is one observation; this is not a claim of unique products.
All displayed results are aggregate counts or common numeric timestamp tokens;
no product descriptions, IDs, URLs, or label examples are printed.

**Evidence checked:** source CSVs; `docs/data_contract.md`; profiling and cleaning
direction notes; current rule decisions; package and notebook source. None of the
available source documentation specifies the timestamp epoch or timezone.
Microsoft documents both [1900 and 1904 Excel date systems](https://support.microsoft.com/en-us/excel/date-systems-in-excel).
Their offset is 1,462 days. Plausible-looking dates alone do not confirm either system.


In [1]:
from collections import Counter
import csv
from datetime import datetime, timedelta
from decimal import Decimal, InvalidOperation
import hashlib
import json
from pathlib import Path
import re

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
RUN = ROOT / 'data/processed/integration-20260930-v1'
summary = json.loads((RUN / 'summary.json').read_text())
def records(path):
    with path.open(encoding='utf-8-sig', newline='') as stream:
        yield from csv.DictReader(stream, strict=True)
def sha256(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()
def missing(value):
    return value.strip().lower() in {'', 'null'}

decisions = {(r['dataset'], int(r['source_row'])): r['status']
             for r in records(RUN / 'row_decisions.csv')}
audit = {}
for dataset in ('training', 'target'):
    source = ROOT / 'data/provided' / summary['sources'][dataset]['file']
    assert sha256(source) == summary['sources'][dataset]['sha256']
    counters = {view: Counter() for view in ('all', 'not_quarantined')}
    numbers = {view: [] for view in counters}
    frequent = Counter()
    blank_categories, null_categories = 0, 0
    for number, row in enumerate(records(source), 1):
        value = row['MDM_InsertDateTime'].strip()
        parsed = None
        if not value or value.lower() == 'null':
            kind = 'missing'
        else:
            try:
                parsed = Decimal(value)
                kind = 'finite_numeric' if parsed.is_finite() else 'nonfinite_numeric'
            except InvalidOperation:
                kind = 'url_text' if value.lower().startswith(('http://', 'https://')) else 'other_text'
        views = ['all']
        if decisions[dataset, number] != 'quarantine':
            views.append('not_quarantined')
        for view in views:
            counters[view]['rows'] += 1
            counters[view][kind] += 1
            if kind == 'finite_numeric':
                numbers[view].append(parsed)
                counters[view]['fractional_numeric'] += int(parsed != parsed.to_integral_value())
            if kind == 'other_text' and re.match(r'^\d{4}-\d{2}-\d{2}', value):
                counters[view]['iso_like_text'] += 1
        if kind == 'finite_numeric':
            frequent[value] += 1
        blank_categories += int(not row['ProductCategory'].strip())
        null_categories += int(row['ProductCategory'].strip().lower() == 'null')
    audit[dataset] = {
        'source_sha256': sha256(source),
        'counts': {v: dict(c) for v, c in counters.items()},
        'numeric_ranges': {v: {'min': str(min(vals)), 'max': str(max(vals)), 'distinct': len(set(vals))}
                           for v, vals in numbers.items() if vals},
        'common_numeric_tokens': frequent.most_common(6),
        'productcategory_missing_tokens': {'blank': blank_categories, 'literal_null': null_categories},
    }
    assert audit[dataset]['source_sha256'] == summary['sources'][dataset]['sha256']
print(json.dumps(audit, indent=2))


{
  "training": {
    "source_sha256": "00f1fcd0d45bc4196ea8c0e9fffa63897c8e24d12a7639468928fbd6afff4452",
    "counts": {
      "all": {
        "rows": 84552,
        "finite_numeric": 84150,
        "fractional_numeric": 38730,
        "url_text": 263,
        "missing": 118,
        "other_text": 21
      },
      "not_quarantined": {
        "rows": 84086,
        "finite_numeric": 84086,
        "fractional_numeric": 38709
      }
    },
    "numeric_ranges": {
      "all": {
        "min": "1",
        "max": "877006774521",
        "distinct": 63
      },
      "not_quarantined": {
        "min": "44936",
        "max": "45177.76383",
        "distinct": 12
      }
    },
    "common_numeric_tokens": [
      [
        "44936",
        36118
      ],
      [
        "44967",
        9259
      ],
      [
        "45117.83254",
        7991
      ],
      [
        "45145.83061",
        6083
      ],
      [
        "44998.35338",
        5624
      ],
      [
        "45027.658

## Candidate date interpretations — hypotheses only

For serials at least 61, the modern 1900-system mapping below uses 1899-12-30
to accommodate Excel's historical leap-year behavior. Early serials are excluded
from this illustration. Fractional days are shown to microsecond resolution;
that display does not imply the source measured time that precisely.
No `Z` or timezone offset is attached because the source timezone is unknown.


In [2]:
examples = []
for token, count in audit['training']['common_numeric_tokens']:
    serial = Decimal(token)
    if not Decimal(61) <= serial <= Decimal(100000):
        continue
    duration = timedelta(microseconds=int(serial * Decimal(86400000000)))
    examples.append({'raw': token, 'training_rows': count,
                     'if_excel_1900': (datetime(1899, 12, 30) + duration).isoformat(),
                     'if_excel_1904': (datetime(1904, 1, 1) + duration).isoformat()})
print(json.dumps(examples, indent=2))


[
  {
    "raw": "44936",
    "training_rows": 36118,
    "if_excel_1900": "2023-01-10T00:00:00",
    "if_excel_1904": "2027-01-11T00:00:00"
  },
  {
    "raw": "44967",
    "training_rows": 9259,
    "if_excel_1900": "2023-02-10T00:00:00",
    "if_excel_1904": "2027-02-11T00:00:00"
  },
  {
    "raw": "45117.83254",
    "training_rows": 7991,
    "if_excel_1900": "2023-07-10T19:58:51.456000",
    "if_excel_1904": "2027-07-11T19:58:51.456000"
  },
  {
    "raw": "45145.83061",
    "training_rows": 6083,
    "if_excel_1900": "2023-08-07T19:56:04.704000",
    "if_excel_1904": "2027-08-08T19:56:04.704000"
  },
  {
    "raw": "44998.35338",
    "training_rows": 5624,
    "if_excel_1900": "2023-03-13T08:28:52.032000",
    "if_excel_1904": "2027-03-14T08:28:52.032000"
  },
  {
    "raw": "45027.6581",
    "training_rows": 5413,
    "if_excel_1900": "2023-04-11T15:47:39.840000",
    "if_excel_1904": "2027-04-12T15:47:39.840000"
  }
]


## Integration parsing check

The combined pipeline uses a narrow blank/null policy and canonical one-based
identity. The legacy target command retains pandas default NA parsing and its
zero-based identity. Check whether that adaptation changed target quarantine
membership on these actual exports. This comparison does not approve the IQR
heuristic or establish that it is appropriate for future model evaluation splits.


In [3]:
from catalogiq.cleaning import load_source, clean_training

training = ROOT / 'data/provided' / summary['sources']['training']['file']
legacy = clean_training(load_source(training))
legacy_quarantine = {int(value) + 1 for value in legacy.quarantine['source_row']}
integrated_target_quarantine = {
    int(row['source_row']) for row in records(RUN / 'row_decisions.csv')
    if row['dataset'] == 'training'
    and any(code in {'4', '5', '6'} for code in row['target_reason_codes'].split(';'))}
parsing_comparison = {'legacy_target_quarantine': len(legacy_quarantine),
                      'integrated_target_quarantine': len(integrated_target_quarantine),
                      'added_source_records': sorted(integrated_target_quarantine - legacy_quarantine),
                      'removed_source_records': sorted(legacy_quarantine - integrated_target_quarantine)}
print(json.dumps(parsing_comparison, indent=2))


{
  "legacy_target_quarantine": 4,
  "integrated_target_quarantine": 4,
  "added_source_records": [],
  "removed_source_records": []
}


## Decision

The common values and fractional parts are consistent with Excel-style day
serials. They do not establish the source date system, timezone, or business
meaning of this field. Corrupted rows also contain displaced text or other numbers.

Preserve `MDM_InsertDateTime` exactly and keep it outside the proposed model
features. Ask the source owner to confirm the export date system, timezone, and
whether this is insertion time rather than event time. If confirmed, a future
derived field can use ISO 8601 (`YYYY-MM-DDTHH:MM:SS[.fraction]`, with an offset
only if known). Retain the original token and log conversion failures. No
timestamp-based split or automatic epoch selection is justified by this audit.
